# Goal
We want to build a model that can simply reverse sort a list of input numbers. 

e.g. prompt `[0,2,1]` the model can output `[2,1,0]`

In [11]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## Rewards
we'll first show 10 examples that we'll use to see how well each reward function works

In [12]:
torch.manual_seed(14)
ten_examples = torch.randint(0, 4, (10, 2, 4))

In [13]:
def apply_reward(reward_func):
    for i, (p, r) in enumerate(ten_examples):
        reward = reward_func(p,r)
        print(f'index {i} |reward {reward}| prompt {p.tolist()} | response{r.tolist()}')

### Distance reward

calculates reward based on how many positions in output match up with true sort

In [14]:
def sort_distance_reward(prompt, response):
    assert len(prompt) == len(response)
    ground_truth = sorted(prompt, reverse=True)
    match = sum(1 for x,y in zip(response, ground_truth) if x == y)
    return match

In [15]:
apply_reward(sort_distance_reward)

index 0 |reward 1| prompt [3, 0, 0, 2] | response[3, 1, 2, 2]
index 1 |reward 1| prompt [0, 2, 0, 1] | response[0, 1, 2, 1]
index 2 |reward 1| prompt [3, 2, 3, 0] | response[3, 0, 1, 2]
index 3 |reward 1| prompt [2, 0, 2, 0] | response[1, 1, 0, 1]
index 4 |reward 2| prompt [1, 0, 3, 3] | response[3, 0, 1, 1]
index 5 |reward 0| prompt [3, 0, 1, 3] | response[2, 0, 3, 1]
index 6 |reward 3| prompt [3, 3, 1, 2] | response[3, 2, 2, 1]
index 7 |reward 0| prompt [0, 3, 0, 3] | response[2, 1, 3, 2]
index 8 |reward 2| prompt [3, 1, 1, 3] | response[1, 1, 1, 1]
index 9 |reward 0| prompt [0, 3, 1, 3] | response[1, 1, 3, 1]


## Sort Inclusion reward
1 point for getting the right numbers; 1 point for the the right ordering where adjacencies decrease

In [16]:
def sort_inclusion_ordering_reward(prompt, response):
    assert len(prompt) == len(response)
    inclusion_reward = sum(1 for x in prompt if x in response)
    order_reward = sum(1 for x,y in zip(response[:-1],response[1:]) if x > y)
    return inclusion_reward + order_reward

In [17]:
apply_reward(sort_inclusion_ordering_reward)

index 0 |reward 3| prompt [3, 0, 0, 2] | response[3, 1, 2, 2]
index 1 |reward 5| prompt [0, 2, 0, 1] | response[0, 1, 2, 1]
index 2 |reward 5| prompt [3, 2, 3, 0] | response[3, 0, 1, 2]
index 3 |reward 3| prompt [2, 0, 2, 0] | response[1, 1, 0, 1]
index 4 |reward 5| prompt [1, 0, 3, 3] | response[3, 0, 1, 1]
index 5 |reward 6| prompt [3, 0, 1, 3] | response[2, 0, 3, 1]
index 6 |reward 6| prompt [3, 3, 1, 2] | response[3, 2, 2, 1]
index 7 |reward 4| prompt [0, 3, 0, 3] | response[2, 1, 3, 2]
index 8 |reward 2| prompt [3, 1, 1, 3] | response[1, 1, 1, 1]
index 9 |reward 4| prompt [0, 3, 1, 3] | response[1, 1, 3, 1]


## Customized reward
1 point for getting the right numbers; -1 point for including an errored number;
1 port for the right ordering where adjacenties decrease or are the same since we allow duplicates

In [18]:
def sort_complex_reward(prompt, response):
    assert len(prompt) == len(response)
    inclusion_reward = sum(1 for x in prompt if x in response)
    exclusion_penalty = sum(-1 for x in response if x not in prompt)
    order_reward = sum(1 for x,y in zip(response[:-1],response[1:]) if x >= y)
    return inclusion_reward + exclusion_penalty + order_reward

In [19]:
apply_reward(sort_complex_reward)

index 0 |reward 3| prompt [3, 0, 0, 2] | response[3, 1, 2, 2]
index 1 |reward 5| prompt [0, 2, 0, 1] | response[0, 1, 2, 1]
index 2 |reward 4| prompt [3, 2, 3, 0] | response[3, 0, 1, 2]
index 3 |reward 1| prompt [2, 0, 2, 0] | response[1, 1, 0, 1]
index 4 |reward 6| prompt [1, 0, 3, 3] | response[3, 0, 1, 1]
index 5 |reward 5| prompt [3, 0, 1, 3] | response[2, 0, 3, 1]
index 6 |reward 7| prompt [3, 3, 1, 2] | response[3, 2, 2, 1]
index 7 |reward 1| prompt [0, 3, 0, 3] | response[2, 1, 3, 2]
index 8 |reward 5| prompt [3, 1, 1, 3] | response[1, 1, 1, 1]
index 9 |reward 5| prompt [0, 3, 1, 3] | response[1, 1, 3, 1]


## Policy / Model 
Simple model to take prompt and define response,  the goal of this is to have it reverse the order


In [62]:
class Model(nn.Module):
    def __init__(self, vocab_size, embd_dim, prompt_len, response_len):
        super().__init__()
        self.embd_dim = embd_dim
        self.embedding = nn.Embedding(vocab_size, embd_dim)
        self.encode_weights = nn.Parameter(torch.randn(prompt_len, embd_dim, embd_dim)/ np.sqrt(embd_dim))
        self.decode_weights = nn.Parameter(torch.randn(response_len, embd_dim, embd_dim) / np.sqrt(embd_dim))
        
    def forward(self, prompts):
        embeddings = self.embedding(prompts)

        # encode: (batch, pos, dim1) @ (pos, dim1, dim2) -> (batch, pos, dim2), then sum over pos
        encoded = (embeddings @ self.encode_weights).sum(dim=1)
        
        # decode: (batch, dim2) -> (batch, pos, dim1)
        decoded = (encoded.unsqueeze(1) @ self.decode_weights)
        
        # logits: (batch, pos, dim1) @ (dim1, vocab) -> (batch, pos, vocab)
        logits = decoded @ self.embedding.weight.T

        return logits

## Generate Prompts

In [63]:
prompt_ex = torch.randint(0, 3, (10, 3))
prompt_ex

tensor([[0, 1, 1],
        [0, 1, 1],
        [2, 1, 0],
        [1, 1, 1],
        [2, 2, 0],
        [1, 1, 0],
        [0, 0, 0],
        [1, 2, 1],
        [0, 1, 1],
        [1, 2, 0]])

## Generate responses from model

In [64]:
def generate_responses(prompts, model, num_responses=1):
    logits = model(prompts)
    batch_size = prompts.shape[0]

    logits = logits.reshape(-1, logits.size(-1))
    probs = torch.softmax(logits, dim=-1)
    
    # sample from logits to get responses
    responses = torch.multinomial(probs, num_responses, replacement=True)
    responses = responses.reshape(batch_size, -1, num_responses).permute(0, 2, 1)

    return responses

In [65]:
vocab_size = 5
embedding_dim = 8
prompt_length = 4
response_length = 4

In [66]:
model = Model(
    vocab_size = vocab_size,
    embd_dim = embedding_dim , 
    prompt_len = prompt_length, 
    response_len = response_length
)

In [87]:
torch.manual_seed(14)
prom = torch.tensor([[4,0,1,2]])
resp = generate_responses(prom, model, 10)
resp.shape, resp

(torch.Size([1, 10, 4]),
 tensor([[[3, 2, 1, 3],
          [1, 3, 1, 3],
          [3, 0, 1, 3],
          [1, 0, 1, 3],
          [1, 0, 1, 3],
          [1, 0, 1, 3],
          [4, 3, 3, 3],
          [3, 0, 1, 3],
          [1, 3, 1, 3],
          [1, 3, 1, 3]]]))

## Compute reward

In [89]:
def compute_reward(prompts, responses, reward_fn):
    batch, n_resp, _ = responses.shape
    rewards = torch.empty(batch, n_resp, dtype=torch.float32)
    for i in range(batch):
        for j in range(n_resp):
            rewards[i,j] = reward_fn(prompts[i,:], responses[i,j,:])
            print(rewards[i,j])

In [93]:

compute_reward(prom, resp, sort_complex_reward)

tensor(2.)
tensor(0.)
tensor(1.)
tensor(2.)
tensor(2.)
tensor(2.)
tensor(1.)
tensor(1.)
tensor(0.)
tensor(0.)


## Compute